# 🛠️ Section 6 — Statistiques Bivariées & Corrélation : Explorer les Relations et Comportements Énergétiques

Bienvenue dans le quatrième module d'outils du cours **Data Science & Python** !

Dans **Section 5**, nous avons découvert les bases de **Pandas** : inspection univariée (`describe()`), nettoyage des valeurs manquantes (`ffill()`), conversions d'unités physiques ($W \rightarrow kW \rightarrow kWh$), et indexation temporelle (`DatetimeIndex`, `resample`, `rolling`).

Jusqu'à présent, nous avons surtout examiné chaque variable de façon isolée (analyse univariée). Mais en Data Science, la véritable valeur naît de **l'analyse bivariée et multivariée** : comprendre comment deux ou plusieurs grandeurs interagissent entre elles.

---

### 🏠 Le Cas d'Étude : Quelles relations régissent la consommation du foyer ?
Dans notre maison connectée, les capteurs enregistrent simultanément plusieurs flux physiques et contextuels. L'analyse bivariée permet de répondre à des questions métier capitales :
1. **La physique de la ligne électrique** : Pourquoi la tension du réseau électrique ($V$) s'effondre-t-elle lorsque la maison tire une forte puissance ($W$) ? La relation est-elle linéaire ou non-linéaire ?
2. **L'impact de la météo** : La température extérieure ($°C$) influence-t-elle la puissance appelée par les appareils (réfrigérateur, ventilation, chauffe-eau) ?
3. **Le contraste Semaine vs Week-end** : Pourquoi le comportement du foyer change-t-il radicalement entre les jours ouvrés (rythme rigide, double pointe matin/soir) et le week-end (rythme étalé et déstructuré) ?
4. **La fragilité face aux valeurs aberrantes (*Outliers*)** : Comment un unique pic extrême de puissance peut-il déformer nos coefficients de corrélation et biaiser les conclusions ?

---

### 🗺️ Votre parcours d'apprentissage dans ce notebook :
0. **👁️ Coup d'œil visuel initial** — Le nuage de points et la comparaison visuelle Semaine vs Week-end
1. **Section 1** — Fondements de la Corrélation Bivariée : Pearson ($r$) et Signification Statistique ($p$-value)
2. **Section 2** — Linéarité vs Monotonie : Pearson vs Spearman (La physique de la tension)
3. **Section 3** — Matrice de Corrélation & Heatmap : Cartographier toutes les relations
4. **Section 4** — Comportements Différenciés : Semaine vs Week-end (Deux modes de vie)
5. **Section 5** — L'Impact des Valeurs Aberrantes (*Outliers*) sur la Corrélation
6. **Section 6** — Visualisations Bivariées Avancées (Régression linéaire et boxplots horaires)
7. **Section 7** — 🏆 Défi Final : L'Enquête Statistique sur le Foyer
8. **Section 8** — Synthèse & Tremplin vers la Modélisation (Machine Learning)


In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display
from scipy.stats import pearsonr, spearmanr

# Configuration graphique soignée
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (11, 4.5)
plt.rcParams['font.size'] = 10

# Relances humoristiques adaptées à l'analyse bivariée & corrélation
_RELANCES_HUMOUR = [
    "Attention : corrélation n'implique JAMAIS causalité !",
    "Un outlier a détourné ta droite de régression !",
    "As-tu confondu corrélation linéaire (Pearson) et corrélation de rang (Spearman) ?",
    "Vérifie bien ta p-value : si p > 0.05, la corrélation n'est pas statistiquement significative !",
    "Pense à éliminer les NaN avec .dropna() avant de lancer pearsonr !",
    "Même Karl Pearson a dû vérifier ses calculs de covariance deux fois !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-3):
    """
    Vérifie la valeur d'une variable ou d'un coefficient dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, (pd.Series, pd.DataFrame)) and isinstance(valeur, (pd.Series, pd.DataFrame)):
            if attendu.shape == valeur.shape:
                if isinstance(attendu, pd.Series):
                    ok = bool(np.allclose(valeur.dropna().values, attendu.dropna().values, atol=tol))
                else:
                    ok = list(valeur.columns) == list(attendu.columns)
        elif isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and bool(np.allclose(valeur, attendu, atol=tol))
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(float(valeur) - float(attendu)) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(
                (abs(float(v) - float(a)) <= tol if isinstance(a, (int, float)) else v == a)
                for v, a in zip(valeur, attendu)
            )
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement Tool 4 initialisé avec Pandas {pd.__version__} et SciPy.")
print("✅ Fonctions qcm(), check(), pearsonr et spearmanr prêtes à l'emploi.")


---
## 👁️ Coup d'œil visuel initial : Relations physiques et contrastes de vie

Avant de manipuler des formules statistiques, **regardons graphiquement les deux grandes relations qui dominent notre jeu de données** :
1. **La chute de tension en ligne** : Quand la puissance active ($W$) grimpe, la tension du réseau ($V$) chute-t-elle de façon systématique ?
2. **Le choc Semaine vs Week-end** : Les habitudes des occupants en semaine ressemblent-elles à celles du week-end ?

Exécutez le **Programme 2** pour visualiser ces deux réalités :


In [ ]:
# Programme 2 — Visualisation initiale des relations bivariées
# Chargement des données prétraitées
df_init = pd.read_csv("data/conso_maison_capteurs.csv")
df_init['ts'] = pd.to_datetime(df_init['timestamp'])
df_init = df_init.set_index('ts')
df_init['puissance_W'] = df_init['puissance_W'].ffill()
df_init['tension_V'] = df_init['tension_V'].ffill()
df_init['heure'] = df_init.index.hour
df_init['est_weekend'] = df_init.index.dayofweek >= 5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Nuage de points Puissance vs Tension (échantillon de 2 000 points pour lisibilité)
echantillon = df_init.sample(2000, random_state=42)
ax1.scatter(echantillon['puissance_W'], echantillon['tension_V'], color='#457b9d', alpha=0.35, edgecolors='none', s=25)
# Ligne de régression linéaire empirique
m, b = np.polyfit(echantillon['puissance_W'], echantillon['tension_V'], 1)
x_vals = np.linspace(echantillon['puissance_W'].min(), echantillon['puissance_W'].max(), 100)
ax1.plot(x_vals, m * x_vals + b, color='#e63946', linewidth=2.5, label=f"Tendance linéaire (pente = {m:.4f} V/W)")
ax1.set_title("1. Relation Puissance vs Tension (Chute en charge)", fontweight='bold')
ax1.set_xlabel("Puissance active (Watts)")
ax1.set_ylabel("Tension réseau (Volts)")
ax1.legend()

# 2. Courbe de charge comparée : Semaine vs Week-end
prof_sem = df_init.loc[~df_init['est_weekend']].groupby('heure')['puissance_W'].mean()
prof_we  = df_init.loc[df_init['est_weekend']].groupby('heure')['puissance_W'].mean()

ax2.plot(prof_sem.index, prof_sem.values, marker='o', color='#1d3557', linewidth=2.5, label="Semaine (Lun-Ven)")
ax2.plot(prof_we.index, prof_we.values, marker='s', color='#e76f51', linewidth=2.5, linestyle='--', label="Week-end (Sam-Dim)")
ax2.fill_between(prof_sem.index, prof_sem.values, prof_we.values, color='#adb5bd', alpha=0.25, label="Écart de mode de vie")
ax2.set_title("2. Consommation moyenne 24h : Semaine vs Week-end", fontweight='bold')
ax2.set_xlabel("Heure de la journée (0h à 23h)")
ax2.set_ylabel("Puissance moyenne (Watts)")
ax2.set_xticks(range(0, 24, 2))
ax2.legend()

plt.tight_layout()
plt.show()


### 🔍 Ce que révèlent ces deux graphiques :
1. **Graphique de gauche (Puissance vs Tension)** : La pente est nettement **descendante**. Plus la maison appelle de la puissance, plus la tension chute ! Il s'agit d'une corrélation négative forte ($r < 0$) dictée par la loi d'Ohm ($\Delta U = R \cdot I$).
2. **Graphique de droite (Semaine vs Week-end)** : 
   - En **semaine**, la consommation explose le soir entre 18h et 23h (repas, four, lessives, TV, avec un pic à plus de **2 170 W**).
   - Le **week-end**, la consommation du soir est remarquablement basse (moins de 1 000 W), mais l'activité est plus soutenue en pleine journée.
   - Les deux courbes ne se superposent pas du tout : il y a un **déphasage comportemental majeur** !

Comment quantifier précisément ces relations avec des outils mathématiques rigoureux ? C'est le rôle des **coefficients de corrélation**.


---
## 1. Fondements de la Corrélation Bivariée : Pearson ($r$) et Signification Statistique ($p$-value)

### 📐 Le Coefficient de Corrélation Linéaire de Pearson ($r$)
Pour deux variables quantitatives continues $X$ et $Y$, le coefficient de Pearson mesure **la force et la direction de leur relation linéaire** :
$$r = \frac{\sum (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum (X_i - \bar{X})^2 \sum (Y_i - \bar{Y})^2}} \in [-1, +1]$$

- **$r = +1$** : Corrélation positive parfaite (si $X$ augmente, $Y$ augmente proportionnellement).
- **$r = -1$** : Corrélation négative parfaite (si $X$ augmente, $Y$ diminue proportionnellement).
- **$r \approx 0$** : Absence de relation linéaire (attention, une relation non-linéaire peut exister !).

### 🎯 Échelle d'interprétation de la force de la relation :
- $|r| \ge 0.7$ : Relation **forte**
- $0.3 \le |r| < 0.7$ : Relation **modérée**
- $|r| < 0.3$ : Relation **faible ou négligeable**

---

### 🎲 Le Test Statistique et la $p$-value : Le résultat est-il dû au hasard ?
Un coefficient $r = 0.45$ calculé sur 10 mesures peut être un simple coup de chance statistique. Pour savoir si la relation observée est généralisable, on formule un **test d'hypothèse** :
- **Hypothèse nulle ($H_0$)** : Il n'y a aucune corrélation réelle dans la population ($r = 0$).
- **La $p$-value** : C'est la probabilité d'observer un coefficient aussi extrême que le nôtre si l'hypothèse nulle était vraie.
- **Seuil de significativité conventionnel $\alpha = 0.05$ (5 %)** :
  - Si **$p\text{-value} \le 0.05$** : Le résultat est **statistiquement significatif** $\rightarrow$ On rejette $H_0$ et on valide l'existence de la relation.
  - Si **$p\text{-value} > 0.05$** : Le résultat **n'est pas significatif** $\rightarrow$ On ne peut pas écarter le hasard.

Dans Python, on utilise `scipy.stats.pearsonr(x, y)` qui renvoie le tuple `(r, p_value)`.


In [ ]:
# Programme 3 — QCM 1
qcm(
    "Si l'on obtient un coefficient de corrélation r = -0.85 avec une p-value = 0.0001 entre la puissance et la tension, que peut-on affirmer ?",
    [
        "Il n'y a aucune relation car le coefficient est négatif.",
        "Il existe une relation linéaire forte et statistiquement significative : quand la puissance augmente, la tension a tendance à chuter.",
        "La p-value est trop faible pour qu'on puisse faire confiance au résultat.",
        "La tension est la seule cause de la puissance électrique."
    ],
    correct=1,
    explication="Un coefficient négatif fort (|r| = 0.85 > 0.7) associé à une p-value < 0.05 prouve une corrélation inverse forte et hautement significative. Rappelons cependant que corrélation ne prouve pas causalité sans validation physique !"
)


---
## 2. Linéarité vs Monotonie : Pearson vs Spearman

### ⚠️ Le piège de Pearson : la non-linéarité
Le coefficient de Pearson suppose que la relation est une **droite** ($y = ax + b$).
Si la relation est courbée, exponentielle, ou soumise à un effet de seuil (par exemple, un thermostat qui ne s'enclenche qu'au-delà de 20°C), **Pearson sous-estime lourdement la force de la relation**.

### 🏆 Le Coefficient de Corrélation de Rang de Spearman ($\rho$)
Le coefficient de Spearman ne s'intéresse pas aux valeurs numériques brutes, mais à leur **rang** (classement ordonné du plus petit au plus grand).
Il évalue si la relation est **monotone** : quand $X$ grandit, $Y$ grandit-il (ou diminue-t-il) systématiquement, quelle que soit la forme de la courbe ?

| Propriété | Pearson ($r$) | Spearman ($\rho$) |
|---|---|---|
| **Type de relation mesurée** | Strictement **linéaire** ($y = ax + b$) | **Monotone** (croissante ou décroissante générale) |
| **Échelle de mesure requise** | Intervalle ou Rapport | Ordinale, Intervalle ou Rapport |
| **Sensibilité aux valeurs extrêmes (*Outliers*)** | **Très sensible** (les extrêmes déforment la moyenne) | **Très robuste** (les rangs absorbent les extrêmes) |
| **Fonction Python** | `scipy.stats.pearsonr(x, y)` | `scipy.stats.spearmanr(x, y)` |

---

### ✏️ Exercice 1 — Calcul de Pearson et Spearman sur la ligne électrique

À l'aide des colonnes `puissance_W` et `tension_V` de `df_init` :
1. Calculez le coefficient de corrélation linéaire de Pearson et stockez-le dans `r_puissance_tension`.
2. Calculez le coefficient de corrélation de rang de Spearman et stockez-le dans `rho_puissance_tension`.
3. Stockez la $p$-value du test de Pearson dans `p_val_pearson`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
r_puissance_tension, p_val_pearson = pearsonr(df_init['puissance_W'], df_init['tension_V'])
rho_puissance_tension, p_val_spearman = spearmanr(df_init['puissance_W'], df_init['tension_V'])
```
</details>


In [ ]:
# Programme 4 — Démonstration : Pearson vs Spearman
demo_p, demo_p_val = pearsonr(df_init['puissance_W'], df_init['tension_V'])
demo_rho, demo_rho_val = spearmanr(df_init['puissance_W'], df_init['tension_V'])

print(f"📊 Résultats sur 99 396 relevés :")
print(f"  • Pearson  r   = {demo_p:.4f}  (p-value = {demo_p_val:.2e})")
print(f"  • Spearman rho = {demo_rho:.4f} (p-value = {demo_rho_val:.2e})")

if demo_p_val < 0.05:
    print("✅ La relation inverse entre puissance et tension est hautement significative (p < 0.05).")


In [ ]:
# Programme 5 — Exercice 1
# À toi de jouer : calcule les coefficients de corrélation demandés :

r_puissance_tension = ...
rho_puissance_tension = ...
p_val_pearson = ...


In [ ]:
# Programme 6 — Test Exercice 1
# Exécutez cette cellule pour valider vos réponses
check("r_puissance_tension", round(r_puissance_tension, 4), -0.5051, tol=1e-3)
check("rho_puissance_tension", round(rho_puissance_tension, 4), -0.2983, tol=1e-3)
check("p_val_pearson < 0.05", p_val_pearson < 0.05, True)


---
## 3. Matrice de Corrélation & Heatmap Globale

Dans un projet de Data Science, un jeu de données compte souvent des dizaines de colonnes. Tester chaque paire de variables à la main serait fastidieux.

Pandas simplifie cette démarche grâce à la méthode magique **`.corr()`** qui calcule instantanément la matrice symétrique de toutes les paires :
```python
matrice_corr = df.corr()
```

### ⏱️ L'effet du rééchantillonnage temporel : Éliminer le bruit haute fréquence
Sur des relevés bruts à 10 secondes, le bruit de mesure et les découpages électroniques perturbent les corrélations instantanées ($r = -0.505$).
Mais si l'on agrège les données au pas horaire (`.resample('1h').mean().dropna()`), on observe les véritables tendances de fond du réseau : la corrélation puissance/tension grimpe alors à **$r \approx -0.988$**, révélant une loi physique quasi parfaite !

---

### ✏️ Exercice 2 — Matrice de corrélation horaire

1. Construisez un DataFrame horaire `df_horaire` à partir de `df_init` en sélectionnant `['puissance_W', 'tension_V', 'temperature_ext_C']`, rééchantillonné au pas horaire (`'1h'`), agrégé par moyenne (`.mean()`), puis nettoyé des heures vides avec `.dropna()` :
   `df_horaire = df_init[['puissance_W', 'tension_V', 'temperature_ext_C']].resample('1h').mean().dropna()`
2. Calculez sa matrice de corrélation avec `.corr()` et stockez-la dans `matrice_corr_h`.
3. Extrayez le coefficient de corrélation horaire entre la puissance et la tension dans `corr_puissance_tension_h`.
4. Extrayez le coefficient entre la puissance et la température extérieure dans `corr_puissance_temp_h`.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df_horaire = df_init[['puissance_W', 'tension_V', 'temperature_ext_C']].resample('1h').mean().dropna()
matrice_corr_h = df_horaire.corr()

corr_puissance_tension_h = float(matrice_corr_h.loc['puissance_W', 'tension_V'])
corr_puissance_temp_h = float(matrice_corr_h.loc['puissance_W', 'temperature_ext_C'])
```
</details>


In [ ]:
# Programme 7 — Démonstration : Matrice de corrélation et Heatmap annotée
demo_horaire = df_init[['puissance_W', 'tension_V', 'temperature_ext_C']].resample('1h').mean().dropna()
matrice_demo = demo_horaire.corr()

fig, ax = plt.subplots(figsize=(6.5, 5))
cax = ax.imshow(matrice_demo.values, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)

cols = list(matrice_demo.columns)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=20, ha='right', fontweight='bold')
ax.set_yticklabels(cols, fontweight='bold')

# Annotation des valeurs dans chaque case
for i in range(len(cols)):
    for j in range(len(cols)):
        val = matrice_demo.iloc[i, j]
        ax.text(j, i, f"{val:.3f}", ha='center', va='center',
                color='white' if abs(val) > 0.5 else 'black', fontweight='bold')

ax.set_title("Matrice de Corrélation Horaire (Heatmap)", fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


In [ ]:
# Programme 8 — Exercice 2
# À toi de jouer : calcule la matrice de corrélation horaire et extrais les coefficients :

df_horaire = ...
matrice_corr_h = ...

corr_puissance_tension_h = ...
corr_puissance_temp_h = ...


In [ ]:
# Programme 9 — Test Exercice 2
# Exécutez cette cellule pour valider vos réponses
check("corr_puissance_tension_h", round(corr_puissance_tension_h, 3), -0.988, tol=0.01)
check("corr_puissance_temp_h", round(corr_puissance_temp_h, 3), 0.110, tol=0.02)
check("matrice_corr_h (dimensions)", matrice_corr_h.shape, (3, 3))


---
## 4. Comportements Différenciés : Semaine vs Week-end

### 🎭 L'illusion des données agrégées (Le paradoxe du profil moyen)
Calculer une corrélation globale sur tout le dataset peut masquer des réalités diamétralement opposées.
En sociologie énergétique, le mode de vie des ménages se scinde en deux régimes très contrastés :
- **En semaine (Lundi à Vendredi)** : Les occupants sont soumis à des horaires réguliers (départ au travail/école, retour vers 18h, repas, sommeil). La consommation électrique est **fortement synchronisée avec l'heure de la journée**.
- **Le week-end (Samedi et Dimanche)** : Les horaires sont libres et déstructurés (grasse matinée, déjeuners tardifs, présence aléatoire dans la journée). La corrélation rigide avec l'heure de la journée s'estompe.

### 🧪 Vérification statistique de l'hypothèse :
Si nous calculons la corrélation entre **l'heure de la journée** ($0$ à $23$) et la **puissance appelée** :
- En semaine, nous attendons une relation positive statistiquement significative ($p < 0.05$) tirée par le pic de fin de journée.
- Le week-end, la relation est-elle encore significative, ou le test $p$-value conclura-t-il à l'absence de lien linéaire ?

---

### ✏️ Exercice 3 — Corrélation Heure-Puissance : Semaine vs Week-end

Ajoutez au DataFrame horaire `df_horaire` les colonnes :
- `df_horaire['heure'] = df_horaire.index.hour`
- `df_horaire['est_weekend'] = df_horaire.index.dayofweek >= 5`

Calculez ensuite les corrélations de Pearson entre `'heure'` et `'puissance_W'` pour chaque sous-groupe :
1. `r_semaine, p_semaine = pearsonr(...)` sur les lignes de semaine (`~df_horaire['est_weekend']`)
2. `r_weekend, p_weekend = pearsonr(...)` sur les lignes de week-end (`df_horaire['est_weekend']`)

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
df_horaire['heure'] = df_horaire.index.hour
df_horaire['est_weekend'] = df_horaire.index.dayofweek >= 5

donnees_semaine = df_horaire[~df_horaire['est_weekend']]
donnees_weekend = df_horaire[df_horaire['est_weekend']]

r_semaine, p_semaine = pearsonr(donnees_semaine['heure'], donnees_semaine['puissance_W'])
r_weekend, p_weekend = pearsonr(donnees_weekend['heure'], donnees_weekend['puissance_W'])
```
</details>


In [ ]:
# Programme 10 — QCM 2
qcm(
    "Si l'on obtient p_semaine = 0.00001 (p < 0.05) et p_weekend = 0.097 (p > 0.05) pour la corrélation Heure-Puissance, que doit-on conclure ?",
    [
        "Il n'y a de corrélation ni en semaine ni le week-end car les chiffres diffèrent.",
        "En semaine, la hausse de puissance au fil de la journée est statistiquement significative ; le week-end, on ne peut pas rejeter l'hypothèse nulle d'absence de corrélation linéaire.",
        "Le capteur de puissance est tombé en panne pendant le week-end.",
        "La p-value prouve que les habitants consomment plus le week-end."
    ],
    correct=1,
    explication="Exactement ! En semaine, le rythme de vie impose une dynamique temporelle reproductible (p < 0.05). Le week-end, la dispersion des comportements fait monter la p-value au-dessus du seuil de 5 %, ce qui empêche d'affirmer une corrélation linéaire significative."
)


In [ ]:
# Programme 11 — Démonstration : Comparaison Semaine vs Week-end
df_horaire['heure'] = df_horaire.index.hour
df_horaire['est_weekend'] = df_horaire.index.dayofweek >= 5

r_sem_demo, p_sem_demo = pearsonr(df_horaire[~df_horaire['est_weekend']]['heure'], df_horaire[~df_horaire['est_weekend']]['puissance_W'])
r_we_demo, p_we_demo = pearsonr(df_horaire[df_horaire['est_weekend']]['heure'], df_horaire[df_horaire['est_weekend']]['puissance_W'])

print(f"🏢 En Semaine  : r = {r_sem_demo:.4f}  | p-value = {p_sem_demo:.3e}  --> Significatif ? {p_sem_demo < 0.05}")
print(f"🛋️ Le Week-end : r = {r_we_demo:.4f}  | p-value = {p_we_demo:.3e}  --> Significatif ? {p_we_demo < 0.05}")


In [ ]:
# Programme 12 — Exercice 3
# À toi de jouer : calcule les corrélations pour chaque sous-groupe :

r_semaine = ...
p_semaine = ...

r_weekend = ...
p_weekend = ...


In [ ]:
# Programme 13 — Test Exercice 3
# Exécutez cette cellule pour valider vos réponses
check("r_semaine", round(r_semaine, 3), 0.289, tol=0.02)
check("p_semaine < 0.05", p_semaine < 0.05, True)
check("r_weekend", round(r_weekend, 3), 0.194, tol=0.02)
check("p_weekend > 0.05", p_weekend > 0.05, True)


---
## 5. L'Impact Majeur des Valeurs Aberrantes (*Outliers*) sur la Corrélation

### 💣 Pourquoi un seul outlier peut ruiner une analyse
Le coefficient de Pearson repose sur la moyenne et l'écart quadratique :
$$\sum (X_i - \bar{X})(Y_i - \bar{Y})$$
Si un point s'éloigne très fortement du nuage principal (par exemple un pic extrême de puissance à **7 000 W** ou un glitch de mesure), sa distance au carré prend un poids démesuré et **tire la droite de régression vers lui**.

### 🔍 Deux stratégies classiques face aux outliers :
1. **La suppression après détection (*Filtering*)** : On retire temporairement les points extrêmes pour mesurer la corrélation propre au régime normal.
2. **L'affectation d'une variable indicatrice (*Dummy Variable*)** : On crée une colonne booléenne `is_outlier = 1` si la valeur dépasse un seuil critique, ce qui permet aux modèles de Machine Learning de traiter ce régime à part.

Dans notre maison, les puissances supérieures à **5 000 W** correspondent à des cumuls rares d'appareils de très forte puissance (four + chauffe-eau + plaques enclenchés simultanément).

---

### ✏️ Exercice 4 — Audit de robustesse face aux pics extrêmes

1. Comptez le nombre de relevés dans `df_init` où la puissance dépasse **5 000 W** et stockez ce nombre dans `nb_outliers_5000` :
   `nb_outliers_5000 = int((df_init['puissance_W'] > 5000).sum())`
2. Isolez les données sans ces pics extrêmes dans un sous-DataFrame `df_sans_pics` :
   `df_sans_pics = df_init[df_init['puissance_W'] <= 5000]`
3. Calculez le coefficient de Pearson entre `puissance_W` et `tension_V` sur ce jeu filtré et stockez-le dans `r_sans_pics`.
4. Observez la variation : la corrélation passe de **-0.505** à **-0.393** !

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
nb_outliers_5000 = int((df_init['puissance_W'] > 5000).sum())
df_sans_pics = df_init[df_init['puissance_W'] <= 5000]
r_sans_pics, _ = pearsonr(df_sans_pics['puissance_W'], df_sans_pics['tension_V'])
```
</details>


In [ ]:
# Programme 14 — Démonstration : Effet de levier des outliers
r_brut, _ = pearsonr(df_init['puissance_W'], df_init['tension_V'])
df_filtre = df_init[df_init['puissance_W'] <= 5000]
r_filtre, _ = pearsonr(df_filtre['puissance_W'], df_filtre['tension_V'])

print(f"• Corrélation avec TOUS les points (99 396 pts)       : r = {r_brut:.4f}")
print(f"• Corrélation SANS les pics > 5000 W ({len(df_filtre):,} pts) : r = {r_filtre:.4f}")
print(f"--> Les pics extrêmes amplifiaient artificiellement la corrélation de {abs(r_brut - r_filtre):.4f} !")


In [ ]:
# Programme 15 — Exercice 4
# À toi de jouer : mesure l'impact du filtrage des pics extrêmes :

nb_outliers_5000 = ...
r_sans_pics = ...


In [ ]:
# Programme 16 — Test Exercice 4
# Exécutez cette cellule pour valider vos réponses
check("nb_outliers_5000", nb_outliers_5000, 3744)
check("r_sans_pics", round(r_sans_pics, 4), -0.3934, tol=1e-3)


---
## 6. Visualisations Bivariées Avancées

Pour restituer fidèlement les résultats d'une analyse bivariée, le Data Scientist s'appuie sur deux visualisations reines :
1. **Le nuage de points avec régression conditionnelle** : Visualiser la droite de tendance pour différents sous-groupes (Semaine vs Week-end).
2. **Le Boxplot horaire comparatif** : Décomposer la distribution (médiane, quartiles et valeurs aberrantes) pour chaque heure de la journée.


In [ ]:
# Programme 17 — Tableau de bord bivarié complet
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Régression linéaire conditionnelle : Semaine vs Week-end
sem = df_horaire[~df_horaire['est_weekend']]
we  = df_horaire[df_horaire['est_weekend']]

axes[0].scatter(sem['heure'], sem['puissance_W'], color='#1d3557', alpha=0.45, label="Semaine (relevés horaires)")
axes[0].scatter(we['heure'], we['puissance_W'], color='#e63946', alpha=0.45, marker='s', label="Week-end (relevés horaires)")

# Droites de tendance
m_s, b_s = np.polyfit(sem['heure'], sem['puissance_W'], 1)
m_w, b_w = np.polyfit(we['heure'], we['puissance_W'], 1)
h_range = np.linspace(0, 23, 24)
axes[0].plot(h_range, m_s * h_range + b_s, color='#1d3557', linewidth=2.5, label=f"Tendance Semaine (pente = +{m_s:.1f} W/h)")
axes[0].plot(h_range, m_w * h_range + b_w, color='#e63946', linewidth=2.5, linestyle='--', label=f"Tendance Week-end (pente = +{m_w:.1f} W/h)")

axes[0].set_title("1. Puissance vs Heure : Régression conditionnelle", fontweight='bold')
axes[0].set_xlabel("Heure de la journée (0h à 23h)")
axes[0].set_ylabel("Puissance moyenne horaire (W)")
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

# 2. Boxplot comparatif par tranche de 4 heures
df_horaire['tranche_4h'] = (df_horaire['heure'] // 4) * 4
labels_tranches = [f"{t}h-{t+4}h" for t in sorted(df_horaire['tranche_4h'].unique())]

data_sem = [df_horaire[(~df_horaire['est_weekend']) & (df_horaire['tranche_4h'] == t)]['puissance_W'].values for t in sorted(df_horaire['tranche_4h'].unique())]
data_we  = [df_horaire[(df_horaire['est_weekend']) & (df_horaire['tranche_4h'] == t)]['puissance_W'].values for t in sorted(df_horaire['tranche_4h'].unique())]

positions = np.arange(len(labels_tranches))
width = 0.35

b1 = axes[1].boxplot(data_sem, positions=positions - width/2, widths=width, patch_artist=True,
                     boxprops=dict(facecolor='#1d3557', color='#1d3557', alpha=0.7),
                     medianprops=dict(color='yellow', linewidth=2))
b2 = axes[1].boxplot(data_we, positions=positions + width/2, widths=width, patch_artist=True,
                     boxprops=dict(facecolor='#e63946', color='#e63946', alpha=0.7),
                     medianprops=dict(color='yellow', linewidth=2))

axes[1].set_xticks(positions)
axes[1].set_xticklabels(labels_tranches)
axes[1].set_title("2. Dispersion par tranche horaire (Semaine vs Week-end)", fontweight='bold')
axes[1].set_xlabel("Créneau horaire")
axes[1].set_ylabel("Puissance moyenne (W)")
axes[1].legend([b1["boxes"][0], b2["boxes"][0]], ['Semaine', 'Week-end'], loc='upper left')

plt.tight_layout()
plt.show()


---
## 7. 🏆 Défi Final : L'Enquête Statistique sur le Foyer

Vous devez rendre un rapport d'audit bivarié confirmant statistiquement les habitudes de vie et la signature électrique du logement.
Répondez aux 4 questions clés du rapport d'enquête :

1. **`r_profil_24h`** : Quel est le coefficient de corrélation de Pearson entre la courbe de charge moyenne 24h de semaine (`prof_sem`) et celle du week-end (`prof_we`) ? (`round(float(pearsonr(prof_sem, prof_we)[0]), 4)`).
   *(Indice : une valeur positive modérée autour de 0.30 prouve que les deux profils partagent le calme de la nuit, mais divergent fortement le reste du temps !)*
2. **`heure_max_diff`** : À quelle heure précise de la journée ($0$ à $23$) observe-t-on le plus grand écart absolu de puissance moyenne entre semaine et week-end ? (`int(abs(prof_sem - prof_we).idxmax())`).
3. **`pct_pics_extremes`** : Quel est le pourcentage (%) exact de relevés où la maison a tiré plus de **4 500 W** ? (`float((df_init['puissance_W'] > 4500).mean() * 100)` arrondi à 2 décimales).
4. **`r_temperature_puissance_hc`** : Pendant les **Heures Creuses (HC)**, quelle est la corrélation de Pearson entre la température extérieure (`temperature_ext_C`) et la puissance (`puissance_W`) ?

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
prof_sem = df_init.loc[~df_init['est_weekend']].groupby('heure')['puissance_W'].mean()
prof_we  = df_init.loc[df_init['est_weekend']].groupby('heure')['puissance_W'].mean()

r_profil_24h = round(float(pearsonr(prof_sem, prof_we)[0]), 4)
heure_max_diff = int(abs(prof_sem - prof_we).idxmax())

pct_pics_extremes = round(float((df_init['puissance_W'] > 4500).mean() * 100), 2)

df_hc = df_init[df_init['periode_tarifaire'] == 'HC']
r_temperature_puissance_hc = round(float(pearsonr(df_hc['temperature_ext_C'], df_hc['puissance_W'])[0]), 4)
```
</details>


In [ ]:
# Programme 18 — Défi Final
# À toi de jouer : réponds aux 4 questions de l'enquête statistique :

r_profil_24h = ...
heure_max_diff = ...
pct_pics_extremes = ...
r_temperature_puissance_hc = ...


In [ ]:
# Programme 19 — Test Défi Final
# Exécutez cette cellule pour valider l'ensemble de votre enquête bivariée !
check("1. Corrélation profils 24h (Semaine vs WE)", r_profil_24h, 0.2987, tol=0.01)
check("2. Heure de divergence maximale (h)", heure_max_diff, 23)
check("3. Pourcentage de pics extrêmes (%)", pct_pics_extremes, 4.41, tol=0.1)
check("4. Corrélation Temp-Puissance en HC", r_temperature_puissance_hc, 0.2425, tol=0.01)


---
## 🎓 Synthèse du Section 6 : Ce que vous maîtrisez désormais

Félicitations pour avoir franchi ce cap dans l'analyse de données avancée !

1. **La corrélation de Pearson ($r$) et sa signification ($p$-value)** : Vous savez quantifier l'intensité d'une relation linéaire et vérifier si elle est statistiquement significative au seuil $\alpha = 0.05$.
2. **Le choix critique entre Pearson et Spearman** : Vous savez identifier quand une relation est non-linéaire ou susceptible d'être faussée par des valeurs extrêmes.
3. **La matrice de corrélation (`df.corr()`) et la Heatmap** : Vous savez cartographier en un coup d'œil l'ensemble des interactions au sein d'un système multi-capteurs.
4. **L'analyse en sous-groupes (Semaine vs Week-end)** : Vous comprenez pourquoi agréger aveuglément des données peut masquer des comportements opposés et savez segmenter vos analyses.
5. **La gestion des valeurs aberrantes (*Outliers*)** : Vous avez mesuré concrètement comment les pics extrêmes peuvent déformer un coefficient statistique et savez créer des variables d'identification (*dummy* `is_outlier`).

---

### 🚀 Tremplin vers la suite du cours :
Dans **Section 7 (Le Réfrigérateur)**, **Partie 2 (Features & Autocorrélation)** et **Partie 3 (La Maison Complète)**, vous combinerez ces statistiques bivariées pour concevoir des algorithmes de Machine Learning capables d'isoler automatiquement chaque appareil électrique dans le signal global de la maison (*NILM*).
